# Entrenamiento de mT5 con otra semilla — versión Kaggle: chileno → español neutro

Ajusta `google/mt5-small` con `data/interim/borrador_entrenamiento.csv` (los mismos 10.490 pares de las otras corridas) y lo evalúa en los 5 conjuntos de prueba. Es el mismo mT5 de `models/mt5/mt5.ipynb` con lote 4 (como las corridas locales), pero con el nombre de salida según la semilla, para repetir mT5 con las semillas 2 y 3.

## Cómo correrlo en Kaggle
1. Entra a [kaggle.com](https://www.kaggle.com), inicia sesión y verifica el teléfono (lo pide para activar GPU e Internet).
2. *Create → New Notebook* y luego *File → Import Notebook* y sube este archivo (`mt5_semillas_kaggle.ipynb`).
3. En el panel derecho (*Session options*): **Accelerator → GPU T4 x2** (el código usa una sola; no elijas P100: las versiones nuevas de PyTorch pueden no tener kernels para ella) y **Internet → On**.
4. En la celda de configuración deja `SEMILLA = 2` (o pon 3 en otra sesión).
5. *Run All*. Tarda unas 1–2 horas (estimación, no medida). Con *Save Version → Save & Run All (Commit)* corre en segundo plano aunque cierres la pestaña.

## Qué queda al terminar (carpeta `/kaggle/working/salida/`)
| Contenido | Ruta |
|---|---|
| Predicciones (lo que necesitamos) | `data/processed/predicciones_mt5_flo_fp32_s<SEMILLA>.csv` |
| BLEU y chrF | `resultados/comparacion_mt5_semillas.csv` |
| Pesos y tokenizer (~1,2 GB, opcional) | `models/mt5_flo_fp32_s<SEMILLA>/` |

Descarga el CSV de predicciones desde la pestaña *Output* de la página del notebook y ponlo en `data/processed/` del repo. **Hazlo antes de cerrar la sesión:** lo que está en `/kaggle/working` se pierde si no guardas la versión.

Si la sesión se corta a media época, vuelve a ejecutar todo con la misma semilla: retoma desde la última época terminada mientras la sesión no se haya reiniciado.

## 1. Configuración

In [ ]:
RAMA = "entrega-final"  # rama de GitHub que trae los datos de entrenamiento
URL_REPO = "https://github.com/carlosiskindacool/proyecto-nlp.git"
MODELO = "google/mt5-small"

PRECISION = "fp32"  # la T4 no tiene bf16 por hardware, y T5/ByT5 se vuelven inestables en fp16: usar fp32.
EPOCAS = 8
LOTE = 4            # igual que las corridas locales de mT5
LR = 5e-4           # igual que mT5, para que la comparación sea justa
VAL = 0.05
SEMILLA = 2         # 2 o 3 (la 1 ya existe): una por sesión
MAX_LONG = 128      # tokens de subpalabra, como en el mT5 original
REANUDAR = True     # retoma desde la última época guardada si la sesión se cayó
SOBRESCRIBIR = False  # True para reemplazar pesos finales ya guardados en Drive

ENTRENAMIENTO = "data/interim/borrador_entrenamiento.csv"
PARES = ["annotation/lote1.csv", "annotation/lote_1_noticias.csv", "annotation/lote_1_tuits.csv",
         "annotation/lote_1_2_noticias.csv", "annotation/lote_1_2_tuits.csv"]

if PRECISION not in ("bf16", "fp32"):
    raise ValueError('PRECISION debe ser "bf16" o "fp32"')

NOMBRE = f"mt5_flo_{PRECISION}" + ("" if SEMILLA == 1 else f"_s{SEMILLA}")
CARPETA_DRIVE = "/kaggle/working/salida"
DIR_PESOS = f"{CARPETA_DRIVE}/models/{NOMBRE}"
DIR_ESTADO = f"{DIR_PESOS}_estado"
RUTA_PREDICCIONES = f"{CARPETA_DRIVE}/data/processed/predicciones_{NOMBRE}.csv"
RUTA_RESUMEN = f"{CARPETA_DRIVE}/resultados/comparacion_mt5_semillas.csv"
DIR_LOCAL = f"/kaggle/working/{NOMBRE}_mejor"
REPO = "/kaggle/working/proyecto-nlp"

## 2. Preparar las carpetas de salida (en Kaggle no hay Drive)

In [ ]:
import os
import shutil

for carpeta in (os.path.dirname(DIR_PESOS), os.path.dirname(RUTA_PREDICCIONES), os.path.dirname(RUTA_RESUMEN)):
    os.makedirs(carpeta, exist_ok=True)
if os.path.exists(f"{DIR_PESOS}/model.safetensors") and not SOBRESCRIBIR:
    raise FileExistsError(f"Ya hay pesos en {DIR_PESOS}. Usa SOBRESCRIBIR = True o cambia SEMILLA.")
libre_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Salida en {CARPETA_DRIVE} | libre: {libre_gb:.1f} GB")
if libre_gb < 6:
    print("Aviso: conviene tener al menos 6 GB libres.")

## 3. Instalar dependencias y traer el repositorio

In [3]:
!pip install -q "transformers==5.24.0" sentencepiece sacrebleu

In [4]:
import subprocess
import sys

if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, URL_REPO, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src/chilenismos")

## 4. Preparar GPU, precisión y datos

In [5]:
import csv
import json
import math
import random

import sacrebleu
import torch
from transformers import Adafactor, AutoModelForSeq2SeqLM, AutoTokenizer

from baselines import cargar_pares, guardar_resumen, limpiar

random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev != "cuda":
    raise RuntimeError("No hay GPU. Activa la T4 en Entorno de ejecución → Cambiar tipo de entorno.")

USAR_BF16 = PRECISION == "bf16" and dev == "cuda"
print("dispositivo:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", "| precisión:", PRECISION)

dispositivo: cuda Tesla T4 | precisión: fp32


In [6]:
def cargar_entrenamiento(ruta, prohibidas):
    with open(ruta, encoding="utf-8") as f:
        filas = list(csv.DictReader(f, delimiter=";"))
    vistas, pares = set(), []
    for r in filas:
        o, n = limpiar(r["oracion"]), limpiar(r["neutro"])  # sin menciones ni emojis, igual que los conjuntos de prueba
        if not o or not n or n.upper() == "DESCARTAR" or o in prohibidas or o in vistas:
            continue
        vistas.add(o)
        pares.append((o, n))
    return pares


conjuntos = {ruta: cargar_pares(ruta) for ruta in PARES}
prohibidas = {p["chileno"] for ps in conjuntos.values() for p in ps}  # evita fuga hacia la evaluación
pares = cargar_entrenamiento(ENTRENAMIENTO, prohibidas)
random.shuffle(pares)
n_val = max(1, round(len(pares) * VAL))
val, train = pares[:n_val], pares[n_val:]
print(f"entrenamiento: {len(train)} pares | validación: {len(val)} | prueba: {sum(map(len, conjuntos.values()))}")

entrenamiento: 1416 pares | validación: 75 | prueba: 450


## 5. Entrenar

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
mod = AutoModelForSeq2SeqLM.from_pretrained(MODELO).to(dev)
opt = Adafactor(mod.parameters(), lr=LR, relative_step=False, scale_parameter=False, warmup_init=False)
mod.gradient_checkpointing_enable()
mod.config.use_cache = False


def codificar(lote):
    enc = tok([x[0] for x in lote], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LONG).to(dev)
    lab = tok(text_target=[x[1] for x in lote], return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_LONG).input_ids.to(dev)
    lab[lab == tok.pad_token_id] = -100
    return enc, lab


def guardar_mejor():
    mod.config.use_cache = True
    mod.save_pretrained(DIR_LOCAL)
    tok.save_pretrained(DIR_LOCAL)
    mod.config.use_cache = False


RUTA_ESTADO = f"{DIR_ESTADO}/estado.pt"
RUTA_MEJOR_DRIVE = f"{DIR_ESTADO}/mejor"
mejor, mejor_epoca, historial, ep_ini = float("inf"), None, [], 1
shutil.rmtree(DIR_LOCAL, ignore_errors=True)
if REANUDAR and os.path.exists(RUTA_ESTADO):
    est = torch.load(RUTA_ESTADO, map_location=dev, weights_only=False)
    mod.load_state_dict(est["modelo"])
    opt.load_state_dict(est["opt"])
    mejor, mejor_epoca, historial, ep_ini = est["mejor"], est["mejor_epoca"], est["historial"], est["epoca"] + 1
    if os.path.isdir(RUTA_MEJOR_DRIVE):
        shutil.copytree(RUTA_MEJOR_DRIVE, DIR_LOCAL)
    del est
    print(f"Reanudo: ya hay {ep_ini - 1} épocas hechas (mejor época {mejor_epoca}); sigo desde la {ep_ini}.", flush=True)
os.makedirs(DIR_ESTADO, exist_ok=True)

for ep in range(ep_ini, EPOCAS + 1):
    random.seed(SEMILLA * 1000 + ep)
    mod.train()
    random.shuffle(train)
    total = 0.0
    for i in range(0, len(train), LOTE):
        enc, lab = codificar(train[i:i + LOTE])
        with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            loss = mod(**enc, labels=lab).loss
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item()
        if (i // LOTE) % 100 == 0:
            print(f"  época {ep} paso {i // LOTE}/{len(train) // LOTE} pérdida {loss.item():.3f}", flush=True)

    mod.eval()
    perd = 0.0
    with torch.no_grad():
        for i in range(0, len(val), LOTE):
            b = val[i:i + LOTE]
            enc, lab = codificar(b)
            with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
                perd += mod(**enc, labels=lab).loss.item() * len(b)
    perd /= len(val)
    perd_train = total / max(1, math.ceil(len(train) / LOTE))
    historial.append({"epoca": ep, "perdida_entrenamiento": perd_train, "perdida_validacion": perd})
    print(f"época {ep}: pérdida de entrenamiento {perd_train:.3f} | validación {perd:.3f}", flush=True)
    if perd < mejor:
        mejor, mejor_epoca = perd, ep
        guardar_mejor()
        shutil.rmtree(RUTA_MEJOR_DRIVE, ignore_errors=True)
        shutil.copytree(DIR_LOCAL, RUTA_MEJOR_DRIVE)
    # estado para reanudar si la sesión se cae (se escribe a un temporal y se renombra: nunca queda a medias)
    torch.save({"modelo": mod.state_dict(), "opt": opt.state_dict(), "epoca": ep, "mejor": mejor,
                "mejor_epoca": mejor_epoca, "historial": historial}, RUTA_ESTADO + ".tmp")
    os.replace(RUTA_ESTADO + ".tmp", RUTA_ESTADO)

if mejor_epoca is None:
    raise RuntimeError("La pérdida de validación no fue válida en ninguna época; no se guardó el modelo.")
print(f"Mejor época: {mejor_epoca} (validación {mejor:.3f})")

## 6. Guardar los pesos en Drive

In [ ]:
with open(f"{DIR_LOCAL}/entrenamiento.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": MODELO, "precision": PRECISION, "epocas": EPOCAS, "lote": LOTE, "lr": LR, "val": VAL, "max_long": MAX_LONG,
               "semilla": SEMILLA, "mejor_epoca": mejor_epoca, "historial": historial}, f, ensure_ascii=False, indent=2)

temporal = f"{DIR_PESOS}.tmp"
shutil.rmtree(temporal, ignore_errors=True)
shutil.copytree(DIR_LOCAL, temporal)
if os.path.isdir(DIR_PESOS):
    shutil.rmtree(DIR_PESOS)
os.rename(temporal, DIR_PESOS)

pesos = f"{DIR_PESOS}/model.safetensors"
if os.path.getsize(pesos) != os.path.getsize(f"{DIR_LOCAL}/model.safetensors"):
    raise RuntimeError("La copia de los pesos en Drive quedó incompleta.")
print(f"Pesos guardados en {DIR_PESOS} ({os.path.getsize(pesos) / 1024**3:.2f} GB)")

## 7. Evaluar el modelo guardado

In [ ]:
def traducir(modelo, frases, lote=8):
    salidas = []
    modelo.eval()
    for i in range(0, len(frases), lote):
        enc = tok(frases[i:i + lote], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LONG).to(dev)
        with torch.no_grad(), torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            out = modelo.generate(**enc, max_new_tokens=96, num_beams=4)
        salidas += tok.batch_decode(out, skip_special_tokens=True)
    return salidas


del mod, opt
torch.cuda.empty_cache()
tok = AutoTokenizer.from_pretrained(DIR_PESOS)
modelo_final = AutoModelForSeq2SeqLM.from_pretrained(DIR_PESOS).to(dev)
modelo_final.config.use_cache = True

filas, salidas = [], []
for ruta, ps in conjuntos.items():
    hip = traducir(modelo_final, [p["chileno"] for p in ps])
    ref = [p["neutro"] for p in ps]
    salidas += [(os.path.basename(ruta), p["chileno"], r, p["termino"], h) for p, r, h in zip(ps, ref, hip)]
    bleu = sacrebleu.corpus_bleu(hip, [ref]).score
    chrf = sacrebleu.corpus_chrf(hip, [ref]).score
    print(f"{os.path.basename(ruta):<24} {NOMBRE:<18} BLEU {bleu:5.1f} chrF {chrf:5.1f}")
    filas.append({"sistema": NOMBRE, "escenario": "", "bleu": f"{bleu:.1f}", "chrf": f"{chrf:.1f}", "n_pares": len(ps),
                  "pares": ruta, "nota": f"{MODELO}; {PRECISION}; ajustado con borradores del LLM"})

with open(RUTA_PREDICCIONES, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["conjunto", "chileno", "referencia", "termino", NOMBRE])
    w.writerows(salidas)
guardar_resumen(RUTA_RESUMEN, filas)

## 8. Confirmar la escritura en Drive

In [ ]:
print("Predicciones:", RUTA_PREDICCIONES)
print("Resumen:", RUTA_RESUMEN)
print("Pesos:", DIR_PESOS)
shutil.rmtree(DIR_ESTADO, ignore_errors=True)   # terminó bien: ya no hace falta el estado de reanudación
shutil.rmtree(DIR_LOCAL, ignore_errors=True)
print("Listo. Descarga el CSV de predicciones desde la pestaña Output antes de cerrar la sesión.")